# Reanalysis of published scRNA-seq data using LN's t-test
Here we assess how the commonly used t-test on log-normalized scRNA-seq data for differential expression (DE) impacts biological conclusions drawn from those data, by comparing it with LN's t-test. To do this, we check whether a gene set enrichment analysis (GSEA) on the results of both DE methods are qualitatively different.
We follow the guidelines in the "Single-cell best practices" book (Heumos, L., Schaar, A.C., Lance, C. et al., 2023).

In [1]:
import scanpy as sc
import gseapy as gp

## Pre-processing

In [2]:
adata = sc.read(
    "kang_counts_25k.h5ad", backup_url="https://figshare.com/ndownloader/files/34464122"
)
adata

AnnData object with n_obs × n_vars = 24673 × 15706
    obs: 'nCount_RNA', 'nFeature_RNA', 'tsne1', 'tsne2', 'label', 'cluster', 'cell_type', 'replicate', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.4', 'seurat_clusters'
    var: 'name'
    obsm: 'X_pca', 'X_umap'

In [3]:
# Storing the counts for later use
adata.X = adata.X.toarray()
# Renaming label to condition
adata.obs = adata.obs.rename({"label": "condition"}, axis=1)
adata.obs["group"] = adata.obs.condition.astype("string") + "_" + adata.obs.cell_type.astype("string")

# Normalizing
sc.pp.normalize_total(adata)
adata_norm = adata.copy()
sc.pp.log1p(adata)

In [4]:
celltype_condition = "stim_FCGR3A+ Monocytes"  # 'stimulated_B',  'stimulated_CD8 T', 'stimulated_CD14 Mono'

## DE and GSEA following [sc-best-practices](https://www.sc-best-practices.org/conditions/gsea_pathway.html)

In [5]:
sc.tl.rank_genes_groups(adata, "group", method="t-test", key_added="t-test", corr_method="bonferroni")

In [6]:
df = sc.get.rank_genes_groups_df(adata, celltype_condition, key="t-test")
# Number of significant genes (e.g., FDR < 0.05)
n_significant = (df["pvals_adj"] < 0.05).sum()
print(f"Number of significant genes (FDR < 0.05): {n_significant}")

Number of significant genes (FDR < 0.05): 4687


In [7]:
log1p_rnk = sc.get.rank_genes_groups_df(adata, celltype_condition, key="t-test")
log1p_rnk

,names,scores,logfoldchanges,pvals,pvals_adj
0,IFITM3,123.019180,3.847020,0.000000e+00,0.000000e+00
1,ISG15,119.732079,3.764381,0.000000e+00,0.000000e+00
2,TYROBP,91.894241,3.043695,0.000000e+00,0.000000e+00
3,TNFSF10,87.408890,3.685921,0.000000e+00,0.000000e+00
4,S100A11,85.721817,2.467858,0.000000e+00,0.000000e+00
...,...,...,...,...,...
15701,NPM1,-57.033661,-3.380572,8.037546e-303,1.262377e-298
15702,ITM2A,-57.096916,-6.033978,0.000000e+00,0.000000e+00
15703,OCIAD2,-62.463715,-4.942270,0.000000e+00,0.000000e+00
15704,CD7,-62.999275,-4.852872,0.000000e+00,0.000000e+00


In [8]:
log1p_gsea = gp.prerank(rnk=log1p_rnk[["names", "scores"]], # or rnk = rnk,
                     gene_sets='KEGG_2016',
                     threads=4,
                     min_size=5,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True, # see what's going on behind the scenes
                    )

2026-01-16 12:16:41,105 [WARNING] Duplicated values found in preranked stats: 0.01% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-16 12:16:41,106 [INFO] Parsing data files for GSEA.............................
2026-01-16 12:16:41,114 [INFO] Enrichr library gene sets already downloaded in: /Users/pedroferreira/.cache/gseapy, use local file
2026-01-16 12:16:41,121 [INFO] 0004 gene_sets have been filtered out when max_size=1000 and min_size=5
2026-01-16 12:16:41,122 [INFO] 0289 gene_sets used for further statistical testing.....
2026-01-16 12:16:41,122 [INFO] Start to run GSEA...Might take a while..................
2026-01-16 12:16:43,969 [INFO] Congratulations. GSEApy runs successfully................



In [9]:
log1p_gsea.res2d[log1p_gsea.res2d["FDR q-val"] < 0.05].sort_values("FDR q-val")

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Ribosome Homo sapiens hsa03010,-0.873782,-2.658311,0.0,0.0,0.0,84/126,4.97%,RPL13;RPL3;RPL5;RPS3A;RPL7;RPS3;RPLP0;RPL13A;R...
1,prerank,Phagosome Homo sapiens hsa04145,0.634326,2.493972,0.0,0.0,0.0,51/124,8.15%,FCGR3A;CTSS;HLA-DPA1;HLA-E;HLA-C;HLA-DRA;HLA-D...
2,prerank,Staphylococcus aureus infection Homo sapiens h...,0.738026,2.349455,0.0,0.0,0.0,20/43,6.33%,FCGR3A;C3AR1;HLA-DPA1;CFD;HLA-DRA;HLA-DRB1;HLA...
3,prerank,Leishmaniasis Homo sapiens hsa05140,0.641486,2.29288,0.0,0.0,0.0,23/65,4.70%,FCGR3A;HLA-DPA1;HLA-DRA;HLA-DRB1;CYBA;NCF1;HLA...
4,prerank,Tuberculosis Homo sapiens hsa05152,0.595008,2.26937,0.0,0.0,0.0,42/148,5.80%,FCGR3A;FCER1G;CTSS;HLA-DPA1;CD74;HLA-DRA;HLA-D...
5,prerank,RIG-I-like receptor signaling pathway Homo sap...,0.663737,2.268581,0.0,0.0,0.0,11/52,4.20%,ISG15;CXCL10;IRF7;DDX58;IFIH1;TANK;NFKBIA;DHX5...
6,prerank,Influenza A Homo sapiens hsa05164,0.571049,2.254096,0.0,0.0,0.0,39/144,6.10%,TNFSF10;CXCL10;IRF7;OAS1;MX1;HLA-DPA1;RSAD2;HL...
15,prerank,T cell receptor signaling pathway Homo sapiens...,-0.663031,-1.967567,0.0,0.0,0.0,25/96,8.86%,CD3D;LAT;CD3E;CD247;LCK;CD8A;CD3G;FYN;CD8B;JUN...
20,prerank,Primary immunodeficiency Homo sapiens hsa05340,-0.72165,-1.864955,0.0,0.000743,0.004,13/32,5.87%,CD3D;CD3E;LCK;IL7R;CD8A;CD8B;CD79A;ICOS;TNFRSF...
19,prerank,Ribosome biogenesis in eukaryotes Homo sapiens...,-0.648363,-1.868378,0.0,0.00099,0.004,31/73,15.03%,NOP58;NHP2;IMP3;NHP2L1;POP5;SBDS;NOB1;RPP40;WD...


## DE and GSEA using LN's t-test

In [16]:
import importlib
import os

file_path = os.path.join(os.getcwd(), "../../pkg/", "scanpy_wrapper.py")
spec = importlib.util.spec_from_file_location("scanpy_wrapper", file_path)
scanpy_wrapper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scanpy_wrapper)

# LN's t-test uses normalized data without log1p
scanpy_wrapper.rank_genes_groups_ln(adata_norm, "group", key_added="lnt-test", groups=[celltype_condition])


Processing group: stim_FCGR3A+ Monocytes


In [17]:
df = sc.get.rank_genes_groups_df(adata_norm, celltype_condition, key="lnt-test")
# Number of significant genes (e.g., FDR < 0.05)
n_significant = (df["pvals_adj"] < 0.05).sum()
print(f"Number of significant genes (FDR < 0.05): {n_significant}")

Number of significant genes (FDR < 0.05): 2161


In [18]:
ln_rnk = sc.get.rank_genes_groups_df(adata_norm, celltype_condition, key="lnt-test")
ln_rnk

,names,scores,logfoldchanges,pvals,pvals_adj
0,MS4A7,95.427521,4.257463,0.0,0.0
1,IFITM3,89.460434,2.614924,0.0,0.0
2,TNFSF10,82.742287,2.827140,0.0,0.0
3,FCGR3A,82.528114,4.222550,0.0,0.0
4,MS4A4A,75.845924,5.423291,0.0,0.0
...,...,...,...,...,...
15701,RPS3,-47.933701,-1.904445,0.0,0.0
15702,RPL10,-47.992931,-1.508253,0.0,0.0
15703,RPS6,-48.056728,-1.929161,0.0,0.0
15704,RPL3,-51.473209,-2.319095,0.0,0.0


In [19]:
ln_gsea = gp.prerank(rnk=ln_rnk[["names", "scores"]], # or rnk = rnk,
                     gene_sets='KEGG_2016',
                     threads=4,
                     min_size=5,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True, # see what's going on behind the scenes
                    )

2026-01-16 12:22:11,226 [WARNING] Duplicated values found in preranked stats: 0.01% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-01-16 12:22:11,227 [INFO] Parsing data files for GSEA.............................
2026-01-16 12:22:11,242 [INFO] Enrichr library gene sets already downloaded in: /Users/pedroferreira/.cache/gseapy, use local file
2026-01-16 12:22:11,257 [INFO] 0004 gene_sets have been filtered out when max_size=1000 and min_size=5
2026-01-16 12:22:11,258 [INFO] 0289 gene_sets used for further statistical testing.....
2026-01-16 12:22:11,259 [INFO] Start to run GSEA...Might take a while..................
2026-01-16 12:22:14,120 [INFO] Congratulations. GSEApy runs successfully................



In [20]:
ln_gsea.res2d[ln_gsea.res2d["FDR q-val"] < 0.05].sort_values("FDR q-val")

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Ribosome Homo sapiens hsa03010,-0.913684,-3.07798,0.0,0.0,0.0,85/126,2.88%,RPL13;RPL3;RPS6;RPL10;RPS3;RPL13A;RPS18;RPL7;R...
14,prerank,Natural killer cell mediated cytotoxicity Homo...,0.678268,2.060205,0.0,0.0,0.0,30/104,7.93%,TNFSF10;FCGR3A;TYROBP;FCER1G;BID;HLA-E;PTPN6;F...
12,prerank,Complement and coagulation cascades Homo sapie...,0.795423,2.101348,0.0,0.0,0.0,12/46,5.11%,C3AR1;CFD;SERPINA1;SERPING1;PLAUR;C1QA;C1QB;C2...
11,prerank,Influenza A Homo sapiens hsa05164,0.670746,2.102021,0.0,0.0,0.0,35/144,4.86%,TNFSF10;CXCL10;IRF7;OAS1;HLA-DPA1;CASP1;DDX58;...
10,prerank,Systemic lupus erythematosus Homo sapiens hsa0...,0.714395,2.120084,0.0,0.0,0.0,23/94,5.37%,FCGR3A;CD86;HLA-DPA1;SSB;FCGR1A;CD40;FCGR3B;C1...
9,prerank,Parkinson's disease Homo sapiens hsa05012,-0.633657,-2.121617,0.0,0.0,0.0,69/109,16.43%,COX4I1;NDUFS5;PARK7;NDUFA4;ATP5A1;COX7A2L;NDUF...
8,prerank,RNA transport Homo sapiens hsa03013,-0.614256,-2.157499,0.0,0.0,0.0,87/151,20.75%,PABPC1;EEF1A1;EIF4A2;EIF3E;EIF3F;RAN;EIF3G;RNP...
13,prerank,Oxidative phosphorylation Homo sapiens hsa00190,-0.625019,-2.079896,0.0,0.0,0.0,56/105,12.45%,COX4I1;NDUFS5;NDUFA4;ATP5A1;COX7A2L;NDUFB11;AT...
6,prerank,Spliceosome Homo sapiens hsa03040,-0.66114,-2.202439,0.0,0.0,0.0,87/126,19.19%,HNRNPA1;SNRPD2;HNRNPM;SRSF2;SRSF3;NHP2L1;LSM7;...
5,prerank,Tuberculosis Homo sapiens hsa05152,0.699336,2.217008,0.0,0.0,0.0,44/148,7.04%,FCGR3A;FCER1G;CTSS;HLA-DPA1;BID;RIPK2;FCGR1A;L...
